# RMShell Solve and Postprocess Tutorial

This notebook walks through the current `RMShellModel` API in a little more detail than the examples.

It covers:

1. Building a small shell problem.
2. Creating canonical material and load input groups.
3. Solving for the shell displacement state.
4. Using the postprocessor in a few different ways.
5. Registering a custom postprocessing output from a new form builder.
6. Reusing the postprocessor with an externally supplied displacement field.

In [ ]:
import numpy as np
import csdl_alpha as csdl
import dolfinx
import dolfinx.io
import meshio
import ufl
from mpi4py import MPI

from femo_alpha.rm_shell.rm_shell_model import RMShellModel, PostOutputSpec

recorder = csdl.Recorder(inline=True)
recorder.start()

## Build a small shell model

The example below creates a simple cantilevered quadrilateral shell mesh directly in the notebook. For larger problems you would normally load the mesh from your own geometry/meshing pipeline.

In [ ]:
def clamped_boundary(x):
    return np.less(x[0], 1.0e-12)

nx, ny = 4, 2
xs = np.linspace(0.0, 10.0, nx + 1)
ys = np.linspace(0.0, 2.0, ny + 1)
points = np.array([[x, y, 0.0] for y in ys for x in xs], dtype=float)
cells = []
for j in range(ny):
    for i in range(nx):
        n0 = j * (nx + 1) + i
        n1 = n0 + 1
        n3 = n0 + (nx + 1)
        n2 = n3 + 1
        cells.append([n0, n1, n2, n3])

mesh_path = "./rmshell_tutorial_mesh.xdmf"
meshio.write(mesh_path, meshio.Mesh(points, [("quad", np.array(cells, dtype=np.int64))]))
with dolfinx.io.XDMFFile(MPI.COMM_WORLD, mesh_path, "r") as xdmf:
    mesh = xdmf.read_mesh(name="Grid")

shell = RMShellModel(
    mesh,
    shell_bc_func=clamped_boundary,
    element_wise_material=False,
    solve_direct=True,
    record=False,
)

## Build canonical material and load inputs

`RMShellModel` now uses explicit input factories. The input groups returned by these helpers are the objects passed into `solve(...)` and later reused by the postprocessor.

In [ ]:
nn = shell.nn
thickness = csdl.Variable(value=0.1 * np.ones(nn), name="thickness")
E = csdl.Variable(value=1.0e8 * np.ones(nn), name="E")
nu = csdl.Variable(value=0.3 * np.ones(nn), name="nu")
density = csdl.Variable(value=10.0 * np.ones(nn), name="density")

nodal_pressure = csdl.Variable(value=np.zeros((nn, 3)), name="nodal_pressure")
nodal_pressure.value[:, 2] = 5.0
node_disp = csdl.Variable(value=np.zeros((nn, 3)), name="node_disp")

material = shell.material_inputs.from_isotropic(
    E=E,
    nu=nu,
    thickness=thickness,
    density=density,
)
loads = shell.load_inputs.from_fields(
    nodal_pressure=nodal_pressure,
    node_disp=node_disp,
)

## Solve the shell problem

The solver returns a `ShellState`, which packages the backend choice, the input groups, and the solved displacement variables. That state can be handed directly to the postprocessor.

In [ ]:

state = shell.solve(material=material, loads=loads)

## Default postprocessing bundle

`shell.post.evaluate(...)` returns the default output bundle. This is convenient when you want the same high-level outputs as the examples without choosing them one by one.

In [ ]:
default_outputs = shell.post.evaluate(state=state)

print("Compliance:", default_outputs.compliance.value)
print("Mass:", default_outputs.mass.value)
print("CG:", default_outputs.cg.value)
print("Tip deflection:", np.max(default_outputs.disp_extracted.value[:, 2]))

## Compute only the outputs you need

For larger workflows, the more useful interface is usually `compute(...)` or `compute_many(...)`. These let you request just the quantities you need.

A `ShellPostContext` lets multiple postprocessing calls share the same prepared inputs and cached intermediate results.

In [ ]:
post_context = shell.post.context(state=state)

compliance = shell.post.compute("compliance", context=post_context)
mass_and_cg = shell.post.compute_many(["mass", "cg"], context=post_context)
kinematics = shell.post.kinematics(state=state)

print("Compliance only:", compliance.value)
print("Mass from compute_many:", mass_and_cg.mass.value)
print("CG from compute_many:", mass_and_cg.cg.value)
print("Displacement field shape:", kinematics.disp_extracted.shape)
print("Rotation field shape:", kinematics.rotations.shape)

The grouped helpers are just convenience wrappers around `compute_many(...)`. They are useful when the outputs naturally belong together, for example mass properties or strain quantities.

In [ ]:
mass_props = shell.post.mass_properties(state=state)
strains = shell.post.strains(state=state)

print("Mass helper mass:", mass_props.mass.value)
print("Mass helper cg:", mass_props.cg.value)
print("Mid-strain field shape:", strains.mid_strain.shape)
print("Curvature field shape:", strains.curvature.shape)

## Register a custom postprocessing output

Custom outputs are registered on `shell.post`. If the new quantity comes from a form, the easiest route is to use one of the builder helpers.

Below, `avg_eps_x` computes the area-averaged mid-surface strain component $\varepsilon_{xx}$ over the whole shell. The builder handles the numerator and area normalization internally.

In [ ]:
shell.post.add_scalar_output(
    "avg_eps_x",
    shell.post.builders.average_strain(
        strain_type="mid",
        component="xx",
    ),
    docstring="Area-averaged mid-surface epsilon_xx.",
)

avg_eps_x = shell.post.compute("avg_eps_x", context=post_context)
print("Average epsilon_xx:", avg_eps_x.value)

You can also register alternative versions of built-in ideas. Here is a second p-norm stress output with different aggregation parameters.

In [ ]:
shell.post.add_scalar_output(
    "pnorm_stress_soft",
    shell.post.builders.pnorm_stress(rho=20, m=1.0e-6),
    docstring="Softer stress aggregation than the built-in pnorm_stress output.",
)

custom_scalars = shell.post.compute_many(
    ["pnorm_stress", "pnorm_stress_soft", "avg_eps_x"],
    context=post_context,
)

print("Built-in pnorm_stress:", custom_scalars.pnorm_stress.value)
print("Custom soft pnorm_stress:", custom_scalars.pnorm_stress_soft.value)
print("Average epsilon_xx:", custom_scalars.avg_eps_x.value)

Sometimes you want a quantity that is still best expressed as a form, but does not match any stock builder. In that case, you can register a custom builder directly by returning a `PostOutputSpec`.

The example below defines a new scalar output, `mean_w`, equal to the area-average of the transverse displacement field over the shell mid-surface:

$$
\mathrm{mean\_w} = \frac{\int_{\Omega} u_z\,dx}{\int_{\Omega} 1\,dx}.
$$

This is a true custom postprocessing form: it is assembled by the postprocessor, can be reused with an externally supplied displacement, and does not rely on one of the stock helper builders.

In [ ]:
def mean_transverse_displacement_builder(context):
    w = context.post_fea.inputs_dict["disp_solid"]["function"][2]
    measure = context.region_measure()
    return PostOutputSpec(
        name="",
        kind="derived_from_forms",
        numerator=PostOutputSpec(
            name="",
            kind="scalar_form_builder",
            form=w * measure,
            arguments=["disp_solid"],
        ),
        denominator=PostOutputSpec(
            name="",
            kind="scalar_form_builder",
            form=context.area_form(),
            arguments=["uhat"],
        ),
    )

shell.post.add_output(
    "mean_w",
    mean_transverse_displacement_builder,
    docstring="Area-averaged transverse displacement from a custom UFL form.",
)

mean_w = shell.post.compute("mean_w", context=post_context)
print("Area-averaged transverse displacement:", mean_w.value)

## Reuse the postprocessor with an external displacement

Postprocessing is separate from solving. If you already have a displacement field from another source, you can evaluate the same outputs by providing `material`, `loads`, and `displacement` directly.

In [ ]:
external_outputs = shell.post.evaluate(
    material=material,
    loads=loads,
    displacement=state.disp_solid,
)

print("External compliance:", external_outputs.compliance.value)
print("External tip deflection:", np.max(external_outputs.disp_extracted.value[:, 2]))

## Direct generalized load vectors

If an upstream transfer already provides the generalized shell load vector, you can skip field reconstruction entirely:

```python
load_vector = shell.assemble_generalized_load_vector(
    nodal_pressure=nodal_pressure.value,
    node_disp=node_disp.value,
)
loads_vector = shell.load_inputs.from_vector(
    load_vector=load_vector,
    node_disp=node_disp,
)
state_vector = shell.solve(material=material, loads=loads_vector)
outputs_vector = shell.post.evaluate(state=state_vector)
```

That is the path you want when an external load-transfer operation already computes the generalized right-hand side in shell ordering.